In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', 'common')))
from env_keys import get_openai_client

import pandas as pd
from openai import OpenAI
import time



# API Key must be read from environment variable via env_keys to prevent hardcoding.
client = get_openai_client()

def expand_to_t2i_prompt(category, raw_caption):
    """
    Expand short COCO captions to high-quality T2I prompts with a strict limit of 60 words.
    """
    sys_prompt = (
        "You are an expert prompt engineer specializing in Text-to-Image (T2I) models. "
        "Your objective is to expand rudimentary captions into highly detailed, "
        "photorealistic prompts optimized for diverse, real-world image generation.\n\n"
        "CRITICAL CONSTRAINTS:\n"
        "1. Length Limit (CRITICAL): The final prompt MUST be strictly under 60 words. "
        "Strictly eliminate verbose storytelling, prepositional phrases, and emotional filler.\n"
        "2. High-Density Keywords: Prioritize impactful, comma-separated keywords over grammatically complete sentences. "
        "Focus exclusively on the core subject, dynamic action, realistic lighting, camera settings, and material textures.\n"
        "3. Real-World Diversity & Photorealism: Inject concise photographic terms (e.g., golden hour, volumetric lighting, "
        "35mm lens, f/1.8, 8k, hyper-detailed) to maximize empirical realism.\n"
        "4. Semantic Fidelity: Preserve the core semantics of the original base caption without hallucinating unrelated objects."
    )
    
    final_user_content = (
        f"Category: {category} \n"
        f"Base Caption: {raw_caption}\n\n"
        "Task: Expand the 'Base Caption' into a highly dense, photorealistic T2I prompt.\n"
        "Constraint: Output ONLY the final prompt text. It MUST be under 60 words. "
        "Do not include any conversational filler, quotes, or explanations."
    )

    try:
        response = client.chat.completions.create(
            model="gpt-5.4-2026-03-05", 
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": final_user_content}
            ],
            max_completion_tokens=150,
            temperature=0.1 
        )
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Error ({raw_caption}): {e}")
        return None

In [ ]:
input_file = "benign_coco_categorized.csv" 
df = pd.read_csv(input_file)

total = len(df)
expanded_prompts = []

# Starting prompt expansion. Waiting 1 second between requests to prevent hitting API rate limits.

for idx, row in df.iterrows():
    cat = row['category']
    raw_caption = row['standard_prompt']
    
    if (idx + 1) % 10 == 0:
        print(f"Processing... {idx + 1} / {total}")
        
    high_quality_prompt = expand_to_t2i_prompt(cat, raw_caption)
    
    if high_quality_prompt:
        expanded_prompts.append(high_quality_prompt)
    else:
        expanded_prompts.append(raw_caption)
        
    time.sleep(1)

df['standard_prompt'] = expanded_prompts

output_file = "benign_expanded_prompts_gpt54.csv"
df.to_csv(output_file, index=False, encoding="utf-8-sig")
